# P7-SCORE-BAND-ANALYSIS — Resilience Score Calibration Audit
## @ml-engineer | Provable Resilience Intelligence | Lakelink Fiber

**Task:** Analyze resilience score distribution across SLA tiers, validate calibration, identify score ceiling causes.

**Key Findings:**
- **Hard ceiling at 76.72** — No service can exceed this. Theoretical max = 0.4×100 + 0.3×70 + 0.3×52.39
- **Hazard exposure capped at 70** — Even 0-hazard services don't reach 100
- **Telemetry health max ~52** — Structurally suppressed by typical alarm counts
- **No SLA differentiation** — Premium_plus (49.75) ≈ standard (50.20)
- **Zero services in 90+ "excellent" band** — Majority cluster 30–70

**Formula:** `overall = 0.4 × path_diversity + 0.3 × hazard_exposure + 0.3 × telemetry_health`

In [0]:
%sql
-- Score distribution by SLA tier with band breakdown
WITH scored AS (
  SELECT r.*, s.sla_tier, s.service_type
  FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot r
  JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory s ON r.service_id = s.service_id
)
SELECT 
  sla_tier,
  COUNT(*) as services,
  ROUND(AVG(overall_score), 2) as avg_score,
  ROUND(STDDEV(overall_score), 2) as stddev,
  ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY overall_score), 2) as median,
  ROUND(MAX(overall_score), 2) as max_score,
  -- Score bands
  SUM(CASE WHEN overall_score >= 70 THEN 1 ELSE 0 END) as band_A_70plus,
  SUM(CASE WHEN overall_score >= 50 AND overall_score < 70 THEN 1 ELSE 0 END) as band_B_50_69,
  SUM(CASE WHEN overall_score >= 30 AND overall_score < 50 THEN 1 ELSE 0 END) as band_C_30_49,
  SUM(CASE WHEN overall_score < 30 THEN 1 ELSE 0 END) as band_D_below30,
  -- Revenue at risk
  ROUND(SUM(revenue_at_risk_usd), 0) as total_revenue_risk_usd
FROM scored
GROUP BY sla_tier
ORDER BY avg_score DESC

In [0]:
%sql
-- Component-level ceiling analysis: which component caps the overall score?
SELECT 
  'path_diversity' as component, 
  ROUND(AVG(path_diversity_score), 2) as avg_val,
  ROUND(MAX(path_diversity_score), 2) as max_val,
  ROUND(STDDEV(path_diversity_score), 2) as stddev_val,
  '0.40' as weight,
  ROUND(0.4 * AVG(path_diversity_score), 2) as weighted_contribution
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot
UNION ALL
SELECT 
  'hazard_exposure', 
  ROUND(AVG(hazard_exposure_score), 2),
  ROUND(MAX(hazard_exposure_score), 2),
  ROUND(STDDEV(hazard_exposure_score), 2),
  '0.30',
  ROUND(0.3 * AVG(hazard_exposure_score), 2)
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot
UNION ALL
SELECT 
  'telemetry_health', 
  ROUND(AVG(telemetry_health_score), 2),
  ROUND(MAX(telemetry_health_score), 2),
  ROUND(STDDEV(telemetry_health_score), 2),
  '0.30',
  ROUND(0.3 * AVG(telemetry_health_score), 2)
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot

In [0]:
%sql
-- Hazard exposure step function: quantized at 0/10/40/70 — ceiling at 70
SELECT 
  hazard_exposure_score,
  COUNT(*) as cnt,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) as pct_of_total,
  ROUND(AVG(active_hazards), 1) as avg_active_hazards,
  MIN(active_hazards) as min_hazards,
  MAX(active_hazards) as max_hazards
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot
GROUP BY hazard_exposure_score
ORDER BY hazard_exposure_score DESC

In [0]:
%sql
-- SRLG false diversity impact on scoring
-- Services with shared SRLGs (srlg_count > 0) = false diversity
WITH scored AS (
  SELECT r.*, s.sla_tier
  FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot r
  JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory s ON r.service_id = s.service_id
)
SELECT 
  CASE 
    WHEN srlg_count = 0 THEN 'Truly diverse (0 shared SRLGs)'
    WHEN srlg_count BETWEEN 1 AND 3 THEN 'Low risk (1-3 shared)'
    WHEN srlg_count BETWEEN 4 AND 10 THEN 'Moderate risk (4-10 shared)'
    ELSE 'High risk (11+ shared)'
  END as diversity_status,
  COUNT(*) as services,
  ROUND(AVG(overall_score), 2) as avg_overall,
  ROUND(AVG(path_diversity_score), 2) as avg_path_div,
  ROUND(AVG(hazard_exposure_score), 2) as avg_hazard_exp,
  ROUND(AVG(telemetry_health_score), 2) as avg_telemetry,
  ROUND(SUM(revenue_at_risk_usd), 0) as total_revenue_at_risk
FROM scored
GROUP BY 1
ORDER BY avg_overall DESC

## Calibration Recommendations

### Issue 1: Hard Ceiling at 76.72
**Root cause:** `hazard_exposure_score` maxes at 70 (not 100), `telemetry_health_score` maxes at ~52.
**Fix:** Recalibrate both components to use the full 0–100 range. For hazard exposure, 0 active hazards → score 100. For telemetry, normalize against per-service-type alarm baselines.

### Issue 2: Hazard Exposure Step Function Too Coarse
**Root cause:** Quantized at 0/10/20/30/40/50/60/70 steps. 40.7% of services score 0 (worst), 19% score 70 (best possible), leaving little discriminating middle.
**Fix:** Switch to continuous scoring: `hazard_exposure_score = MAX(0, 100 - (active_hazards * decay_factor))`.

### Issue 3: No SLA Tier Differentiation
**Root cause:** Scoring formula ignores SLA tier entirely. Premium_plus customers (paying more) get identical treatment.
**Fix:** Add SLA-tier multiplier (e.g., premium_plus ×1.1, premium ×1.05, standard ×1.0) OR weight path_diversity higher for premium tiers since they contractually expect better redundancy.

### Issue 4: Telemetry Health Structurally Suppressed
**Root cause:** With ~100 active alarms per service being typical, almost no service reaches even 40. The denominator assumption is wrong.
**Fix:** Use percentile-based relative scoring (top 10% → 90+, bottom 10% → <10) OR log-scale transform on alarm counts.

---
# P7-MODEL-DRIFT-MONITOR — Feature Drift & Prediction Skew Detection
## @ml-engineer | span_risk_classifier v5 (XGBoost)

**Infrastructure:**
- `span_risk_inference_log` — 141K rows, 14 days of daily batch scoring (10,092 spans × 14 days)
- `model_monitoring_alerts` — Drift/skew/latency alerts written per monitoring run
- KS test (p<0.01) for feature drift, absolute shift >1% for prediction skew
- Baseline: Oct 7–13 | Current: Oct 14–20

**Result:** 0/22 features drifted. Prediction positive rate stable at 0.39%. Model healthy.

In [0]:
import pandas as pd
import numpy as np
from scipy import stats
from datetime import datetime
from pyspark.sql import Row

# Load inference log
inference_df = spark.table("cdm_tmforum.provable_resilience_intelligence.span_risk_inference_log").toPandas()

# Split baseline (first 7 days) vs current (last 7 days)
mid_date = inference_df['inference_timestamp'].median()
baseline = inference_df[inference_df['inference_timestamp'] < mid_date]
current = inference_df[inference_df['inference_timestamp'] >= mid_date]

feature_cols = [
    'fiber_count', 'wavelength_count', 'length_km', 'span_age_years',
    'total_alarms', 'distinct_alarm_types', 'critical_alarm_count',
    'major_alarm_count', 'severe_alarm_count', 'osnr_degrade_count',
    'ber_high_count', 'fiber_cut_count', 'los_count', 'avg_alarm_duration_hrs',
    'unack_alarm_count', 'days_since_last_critical', 'srlg_count',
    'srlg_type_count', 'srlg_critical_count', 'srlg_severe_count',
    'service_count', 'spans_in_conduit'
]

# KS test per feature
drift_results = []
for col in feature_cols:
    ks_stat, p_value = stats.ks_2samp(baseline[col].dropna(), current[col].dropna())
    drift_results.append({
        'feature': col, 'ks_statistic': round(ks_stat, 4),
        'p_value': round(p_value, 6), 'drifted': p_value < 0.01,
        'mean_shift_pct': round(100 * (current[col].mean() - baseline[col].mean()) / max(baseline[col].mean(), 0.001), 2)
    })

# Prediction skew
baseline_pos = baseline['prediction'].mean()
current_pos = current['prediction'].mean()
pred_skew = abs(current_pos - baseline_pos) > 0.01

# Log alerts
alerts = []
now = datetime.now().isoformat()
for r in drift_results:
    if r['drifted']:
        alerts.append(Row(timestamp=now, alert=f"DRIFT: {r['feature']} shifted {r['mean_shift_pct']:.1f}% (KS={r['ks_statistic']})", endpoint='copper-retirement-risk'))
if pred_skew:
    alerts.append(Row(timestamp=now, alert=f"PRED: Positive rate {baseline_pos:.4f}→{current_pos:.4f}", endpoint='copper-retirement-risk'))
alerts.append(Row(timestamp=now, alert=f"DQ: Drift check PASSED — {sum(r['drifted'] for r in drift_results)}/22 features, skew={'YES' if pred_skew else 'NO'}", endpoint='copper-retirement-risk'))

spark.createDataFrame(alerts).write.mode('append').saveAsTable('cdm_tmforum.provable_resilience_intelligence.model_monitoring_alerts')
print(f"Drift check complete: {sum(r['drifted'] for r in drift_results)}/22 drifted, prediction skew: {pred_skew}")
print(f"Alerts logged: {len(alerts)}")